# 02 — Plate Feature Engineering

## Plate Value Intelligence

This notebook transforms licence-plate text into explainable, commercially meaningful features for valuation, premium-asset identification, and collectibility analysis.

### Objectives

- Load the governed 2025 valuation dataset
- Standardise plate text
- Extract letter and number structure
- Identify repetition, symmetry, and rarity patterns
- Create interpretable plate-format features
- Prepare a reusable feature table for modelling

### Feature-design principle

Features should be reproducible, explainable, and commercially meaningful. Historical research is used to guide hypotheses, but all relationships will be tested on the 2025 auction data.

In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

/Users/osmanorka/.matplotlib is not a writable directory
Matplotlib created a temporary cache directory at /var/folders/p7/db_fdd8929jc34td4b6dfcg40000gn/T/matplotlib-71cmix1f because there was an issue with the default path ({configdir}); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.
Matplotlib is building the font cache; this may take a moment.


In [2]:
PROJECT_ROOT = Path.cwd()
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

VALUATION_FILE = (
    PROCESSED_DATA_DIR
    / "valuation_core_2025.csv"
)

print(f"Valuation file: {VALUATION_FILE}")
print(f"File exists: {VALUATION_FILE.exists()}")

Valuation file: /Users/osmanorka/Plate-Value-Intelligence/data/processed/valuation_core_2025.csv
File exists: True


In [3]:
df = pd.read_csv(VALUATION_FILE)

print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]}")

Rows: 17,782
Columns: 16


In [4]:
df[
    [
        "event_code",
        "lot_number",
        "plate",
        "hammer_price_gbp",
    ]
].head(20)

Out[0]: 
   event_code  lot_number     plate  hammer_price_gbp
0        B270           1    1984 A          6,510.00
1        B270           2   7777 AA          5,560.00
2        B270           3    AAD 8L          3,710.00
3        B270           4  A881 AKA            480.00
4        B270           5   AAK 64R          3,140.00
5        B270           6   A99 ALB          1,400.00
6        B270           7   AAL 33N            780.00
7        B270           8   249 AAN          1,520.00
8        B270           9  A114 ANA          2,720.00
9        B270          10  A128 ART            360.00
10       B270          11    188 AB          4,260.00
11       B270          12   1964 AB          3,910.00
12       B270          13   ABA 21H            800.00
13       B270          14  AB06 BEY          1,010.00
14       B270          15   ABB 48S          2,210.00
15       B270          16  ABB 188T          1,130.00
16       B270          17   ABB 45X          7,070.00
17       B270      

In [5]:
df["plate"].sample(
    30,
    random_state=42
).tolist()


Out[0]: 
['EAU 7Y',
 'HAN 8M',
 '11 NSC',
 'XXI 92',
 'WGM 4',
 'DAN 629B',
 '40 NAN',
 'W411 EDD',
 'LOU 955B',
 '3 JXR',
 'DAN 87X',
 'COU 870N',
 'RAY 49N',
 'MCU 7S',
 'R466 ATT',
 '1 LVV',
 'RY57 DER',
 'TO53 MMY',
 '75 LAJ',
 'DBR 33N',
 '130 GD',
 '74 NJT',
 'T87 MAS',
 'ABG 7S',
 'SON 156J',
 'A143 ENA',
 '428 GT',
 '8 LJE',
 'L112 ANN',
 'KAY 74S']


## 2. Plate Text Standardisation

Raw plate text is preserved for auditability. A standardised representation is created for repeatable feature engineering.

In [6]:
df_features = df.copy()

df_features["plate_raw"] = (
    df_features["plate"]
    .astype("string")
)

df_features["plate_clean"] = (
    df_features["plate_raw"]
    .str.upper()
    .str.strip()
    .str.replace(
        r"[^A-Z0-9]",
        "",
        regex=True
    )
)

df_features[
    [
        "plate_raw",
        "plate_clean"
    ]
].head(20)

Out[0]: 
   plate_raw plate_clean
0     1984 A       1984A
1    7777 AA      7777AA
2     AAD 8L       AAD8L
3   A881 AKA     A881AKA
4    AAK 64R      AAK64R
5    A99 ALB      A99ALB
6    AAL 33N      AAL33N
7    249 AAN      249AAN
8   A114 ANA     A114ANA
9   A128 ART     A128ART
10    188 AB       188AB
11   1964 AB      1964AB
12   ABA 21H      ABA21H
13  AB06 BEY     AB06BEY
14   ABB 48S      ABB48S
15  ABB 188T     ABB188T
16   ABB 45X      ABB45X
17  ABN 451R     ABN451R
18  ABR 115H     ABR115H
19   A70 BRV      A70BRV


In [7]:
plate_quality_summary = pd.Series(
    {
        "total_rows": len(df_features),
        "missing_raw_plate": (
            df_features["plate_raw"]
            .isna()
            .sum()
        ),
        "missing_clean_plate": (
            df_features["plate_clean"]
            .isna()
            .sum()
        ),
        "empty_clean_plate": (
            df_features["plate_clean"]
            .eq("")
            .sum()
        ),
        "duplicate_clean_plates": (
            df_features["plate_clean"]
            .duplicated()
            .sum()
        ),
    },
    name="value"
)

plate_quality_summary

Out[0]: 
total_rows                17782
missing_raw_plate             0
missing_clean_plate           0
empty_clean_plate             0
duplicate_clean_plates        3
Name: value, dtype: int64


## 3. Core Structural Features

Core features describe the basic length, letter composition, and numeric composition of each plate.

In [8]:
df_features["plate_length"] = (
    df_features["plate_clean"]
    .str.len()
)

df_features["letter_count"] = (
    df_features["plate_clean"]
    .str.count(r"[A-Z]")
)

df_features["digit_count"] = (
    df_features["plate_clean"]
    .str.count(r"[0-9]")
)

df_features["has_letters"] = (
    df_features["letter_count"] > 0
).astype(int)

df_features["has_digits"] = (
    df_features["digit_count"] > 0
).astype(int)

df_features["is_letters_only"] = (
    df_features["digit_count"].eq(0)
).astype(int)

df_features["is_digits_only"] = (
    df_features["letter_count"].eq(0)
).astype(int)

In [9]:
df_features[
    [
        "plate_raw",
        "plate_clean",
        "plate_length",
        "letter_count",
        "digit_count",
        "has_letters",
        "has_digits",
        "is_letters_only",
        "is_digits_only",
    ]
].head(20)

Out[0]: 
   plate_raw plate_clean  plate_length  letter_count  digit_count  \
0     1984 A       1984A             5             1            4   
1    7777 AA      7777AA             6             2            4   
2     AAD 8L       AAD8L             5             4            1   
3   A881 AKA     A881AKA             7             4            3   
4    AAK 64R      AAK64R             6             4            2   
5    A99 ALB      A99ALB             6             4            2   
6    AAL 33N      AAL33N             6             4            2   
7    249 AAN      249AAN             6             3            3   
8   A114 ANA     A114ANA             7             4            3   
9   A128 ART     A128ART             7             4            3   
10    188 AB       188AB             5             2            3   
11   1964 AB      1964AB             6             2            4   
12   ABA 21H      ABA21H             6             4            2   
13  AB06 BEY     AB06BEY 

## 4. Numeric Component Features

Numeric components are extracted to capture low numbers, repeated digits, numeric length, and other value-related patterns.

In [10]:
def extract_numeric_text(plate: str) -> str:
    return "".join(re.findall(r"\d+", plate))


df_features["numeric_text"] = (
    df_features["plate_clean"]
    .apply(extract_numeric_text)
)

df_features["numeric_length"] = (
    df_features["numeric_text"]
    .str.len()
)

df_features["numeric_value"] = pd.to_numeric(
    df_features["numeric_text"],
    errors="coerce"
)

df_features[
    [
        "plate_raw",
        "plate_clean",
        "numeric_text",
        "numeric_length",
        "numeric_value",
    ]
].head(20)

Out[0]: 
   plate_raw plate_clean numeric_text  numeric_length  numeric_value
0     1984 A       1984A         1984               4           1984
1    7777 AA      7777AA         7777               4           7777
2     AAD 8L       AAD8L            8               1              8
3   A881 AKA     A881AKA          881               3            881
4    AAK 64R      AAK64R           64               2             64
5    A99 ALB      A99ALB           99               2             99
6    AAL 33N      AAL33N           33               2             33
7    249 AAN      249AAN          249               3            249
8   A114 ANA     A114ANA          114               3            114
9   A128 ART     A128ART          128               3            128
10    188 AB       188AB          188               3            188
11   1964 AB      1964AB         1964               4           1964
12   ABA 21H      ABA21H           21               2             21
13  AB06 BEY     AB06BEY 

In [11]:
df_features["is_single_digit_number"] = (
    df_features["numeric_value"]
    .between(0, 9, inclusive="both")
    .fillna(False)
    .astype(int)
)

df_features["is_two_digit_number"] = (
    df_features["numeric_value"]
    .between(10, 99, inclusive="both")
    .fillna(False)
    .astype(int)
)

df_features["is_number_below_100"] = (
    df_features["numeric_value"]
    .lt(100)
    .fillna(False)
    .astype(int)
)

df_features["is_number_below_1000"] = (
    df_features["numeric_value"]
    .lt(1000)
    .fillna(False)
    .astype(int)
)

## 5. Repetition and Memorability Features

Repeated letters and digits may improve memorability and perceived collectibility.

In [12]:
def max_character_run(text: str) -> int:
    if not text:
        return 0

    runs = re.findall(r"(.)\1*", text)
    return max(
        len(match.group(0))
        for match in re.finditer(r"(.)\1*", text)
    )


df_features["unique_character_count"] = (
    df_features["plate_clean"]
    .apply(lambda value: len(set(value)))
)

df_features["repeated_character_count"] = (
    df_features["plate_length"]
    - df_features["unique_character_count"]
)

df_features["has_repeated_character"] = (
    df_features["repeated_character_count"] > 0
).astype(int)

df_features["maximum_character_run"] = (
    df_features["plate_clean"]
    .apply(max_character_run)
)

In [13]:
df_features["has_repeated_digit"] = (
    df_features["numeric_text"]
    .apply(
        lambda value: (
            len(value) > 1
            and len(set(value)) < len(value)
        )
    )
    .astype(int)
)

df_features["letter_text"] = (
    df_features["plate_clean"]
    .str.replace(r"[^A-Z]", "", regex=True)
)

df_features["has_repeated_letter"] = (
    df_features["letter_text"]
    .apply(
        lambda value: (
            len(value) > 1
            and len(set(value)) < len(value)
        )
    )
    .astype(int)
)

In [14]:
df_features["is_palindrome"] = (
    df_features["plate_clean"]
    .apply(
        lambda value: (
            len(value) > 1
            and value == value[::-1]
        )
    )
    .astype(int)
)

In [15]:
def contains_sequence(text: str, minimum_length: int = 3) -> int:
    sequences = [
        "0123456789",
        "9876543210",
        "ABCDEFGHIJKLMNOPQRSTUVWXYZ",
        "ZYXWVUTSRQPONMLKJIHGFEDCBA",
    ]

    for sequence in sequences:
        for length in range(minimum_length, len(text) + 1):
            for start in range(len(text) - length + 1):
                fragment = text[start:start + length]

                if fragment in sequence:
                    return 1

    return 0


df_features["has_sequential_pattern"] = (
    df_features["plate_clean"]
    .apply(contains_sequence)
)

In [16]:
def create_plate_pattern(plate: str) -> str:
    pattern = []

    for character in plate:
        if character.isalpha():
            pattern.append("L")
        elif character.isdigit():
            pattern.append("D")
        else:
            pattern.append("O")

    return "".join(pattern)


df_features["plate_pattern"] = (
    df_features["plate_clean"]
    .apply(create_plate_pattern)
)

In [17]:
df_features[
    [
        "plate_raw",
        "plate_clean",
        "plate_pattern",
        "numeric_value",
        "is_number_below_100",
        "has_repeated_digit",
        "has_repeated_letter",
        "maximum_character_run",
        "is_palindrome",
        "has_sequential_pattern",
    ]
].head(30)

Out[0]: 
   plate_raw plate_clean plate_pattern  numeric_value  is_number_below_100  \
0     1984 A       1984A         DDDDL           1984                    0   
1    7777 AA      7777AA        DDDDLL           7777                    0   
2     AAD 8L       AAD8L         LLLDL              8                    1   
3   A881 AKA     A881AKA       LDDDLLL            881                    0   
4    AAK 64R      AAK64R        LLLDDL             64                    1   
5    A99 ALB      A99ALB        LDDLLL             99                    1   
6    AAL 33N      AAL33N        LLLDDL             33                    1   
7    249 AAN      249AAN        DDDLLL            249                    0   
8   A114 ANA     A114ANA       LDDDLLL            114                    0   
9   A128 ART     A128ART       LDDDLLL            128                    0   
10    188 AB       188AB         DDDLL            188                    0   
11   1964 AB      1964AB        DDDDLL           1964  

In [18]:
feature_quality_summary = pd.Series(
    {
        "rows": len(df_features),
        "unique_clean_plates": (
            df_features["plate_clean"].nunique()
        ),
        "plates_below_100": (
            df_features["is_number_below_100"].sum()
        ),
        "plates_with_repeated_digits": (
            df_features["has_repeated_digit"].sum()
        ),
        "plates_with_repeated_letters": (
            df_features["has_repeated_letter"].sum()
        ),
        "palindromes": (
            df_features["is_palindrome"].sum()
        ),
        "sequential_patterns": (
            df_features["has_sequential_pattern"].sum()
        ),
        "unique_plate_patterns": (
            df_features["plate_pattern"].nunique()
        ),
    },
    name="value"
)

feature_quality_summary

Out[0]: 
rows                            17782
unique_clean_plates             17779
plates_below_100                10826
plates_with_repeated_digits      5703
plates_with_repeated_letters     3795
palindromes                         0
sequential_patterns               230
unique_plate_patterns              24
Name: value, dtype: int64


## 7. Repeated Plate Assessment

Repeated plate values are reviewed before modelling. A repeated plate may represent a re-listing across auction events rather than a duplicate extraction error.

In [19]:
repeated_plates = (
    df_features[
        df_features["plate_clean"].duplicated(
            keep=False
        )
    ]
    .sort_values(
        ["plate_clean", "event_code"]
    )
)

repeated_plates[
    [
        "event_code",
        "auction_month",
        "lot_number",
        "plate_raw",
        "plate_clean",
        "hammer_price_gbp",
    ]
]

Out[0]: 
      event_code auction_month  lot_number plate_raw plate_clean  \
3310        B271      February        1367    197 OS       197OS   
17146       B278      November        1356    197 OS       197OS   
673         B270       January         688  GH16 OST     GH16OST   
16485       B278      November         687  GH16 OST     GH16OST   
1589        B270       January        1616  SCO 799H     SCO799H   
13450       B276     September        1618  SCO 799H     SCO799H   

       hammer_price_gbp  
3310           3,510.00  
17146          4,160.00  
673              990.00  
16485          1,030.00  
1589             710.00  
13450            650.00  


## 8. Plate Segment Features

The original spacing structure is retained because the number and length of visible plate segments may contribute to readability and memorability.

In [20]:
df_features["plate_segments"] = (
    df_features["plate_raw"]
    .str.upper()
    .str.strip()
    .str.split()
)

df_features["segment_count"] = (
    df_features["plate_segments"]
    .str.len()
)

df_features["first_segment_length"] = (
    df_features["plate_segments"]
    .apply(
        lambda values: (
            len(values[0])
            if isinstance(values, list)
            and len(values) > 0
            else np.nan
        )
    )
)

df_features["last_segment_length"] = (
    df_features["plate_segments"]
    .apply(
        lambda values: (
            len(values[-1])
            if isinstance(values, list)
            and len(values) > 0
            else np.nan
        )
    )
)

In [21]:
df_features[
    [
        "plate_raw",
        "plate_segments",
        "segment_count",
        "first_segment_length",
        "last_segment_length",
    ]
].head(20)

Out[0]: 
   plate_raw plate_segments  segment_count  first_segment_length  \
0     1984 A      [1984, A]              2                     4   
1    7777 AA     [7777, AA]              2                     4   
2     AAD 8L      [AAD, 8L]              2                     3   
3   A881 AKA    [A881, AKA]              2                     4   
4    AAK 64R     [AAK, 64R]              2                     3   
5    A99 ALB     [A99, ALB]              2                     3   
6    AAL 33N     [AAL, 33N]              2                     3   
7    249 AAN     [249, AAN]              2                     3   
8   A114 ANA    [A114, ANA]              2                     4   
9   A128 ART    [A128, ART]              2                     4   
10    188 AB      [188, AB]              2                     3   
11   1964 AB     [1964, AB]              2                     4   
12   ABA 21H     [ABA, 21H]              2                     3   
13  AB06 BEY    [AB06, BEY]            

In [22]:
def extract_character_blocks(
    plate: str
) -> list[str]:
    return re.findall(
        r"[A-Z]+|\d+",
        plate
    )


df_features["character_blocks"] = (
    df_features["plate_clean"]
    .apply(extract_character_blocks)
)

df_features["block_count"] = (
    df_features["character_blocks"]
    .str.len()
)

df_features["longest_block_length"] = (
    df_features["character_blocks"]
    .apply(
        lambda blocks: (
            max(map(len, blocks))
            if blocks
            else 0
        )
    )
)

## 9. Interpretable Plate Format Classification

Detailed character patterns are grouped into broader, stakeholder-friendly format categories.

In [23]:
def classify_plate_format(
    pattern: str
) -> str:
    if set(pattern) == {"L"}:
        return "letters_only"

    if set(pattern) == {"D"}:
        return "digits_only"

    if re.fullmatch(r"L+D+", pattern):
        return "letters_then_digits"

    if re.fullmatch(r"D+L+", pattern):
        return "digits_then_letters"

    if re.fullmatch(r"L+D+L+", pattern):
        return "letters_digits_letters"

    if re.fullmatch(r"D+L+D+", pattern):
        return "digits_letters_digits"

    return "mixed_complex"


df_features["plate_format_group"] = (
    df_features["plate_pattern"]
    .apply(classify_plate_format)
)

In [24]:
format_distribution = (
    df_features["plate_format_group"]
    .value_counts()
    .rename_axis("plate_format_group")
    .reset_index(name="plate_count")
)

format_distribution

Out[0]: 
       plate_format_group  plate_count
0  letters_digits_letters        12014
1     digits_then_letters         4580
2     letters_then_digits         1188


In [25]:
df_features["all_digits_same"] = (
    df_features["numeric_text"]
    .apply(
        lambda value: (
            len(value) >= 2
            and len(set(value)) == 1
        )
    )
    .astype(int)
)

df_features["all_letters_same"] = (
    df_features["letter_text"]
    .apply(
        lambda value: (
            len(value) >= 2
            and len(set(value)) == 1
        )
    )
    .astype(int)
)

df_features["has_double_character"] = (
    df_features["maximum_character_run"]
    .ge(2)
    .astype(int)
)

df_features["has_triple_character"] = (
    df_features["maximum_character_run"]
    .ge(3)
    .astype(int)
)

df_features["has_four_character_run"] = (
    df_features["maximum_character_run"]
    .ge(4)
    .astype(int)
)

In [26]:
df_features[
    [
        "plate_raw",
        "plate_pattern",
        "plate_format_group",
        "segment_count",
        "block_count",
        "all_digits_same",
        "all_letters_same",
        "has_double_character",
        "has_triple_character",
        "has_four_character_run",
    ]
].head(30)

Out[0]: 
   plate_raw plate_pattern      plate_format_group  segment_count  \
0     1984 A         DDDDL     digits_then_letters              2   
1    7777 AA        DDDDLL     digits_then_letters              2   
2     AAD 8L         LLLDL  letters_digits_letters              2   
3   A881 AKA       LDDDLLL  letters_digits_letters              2   
4    AAK 64R        LLLDDL  letters_digits_letters              2   
5    A99 ALB        LDDLLL  letters_digits_letters              2   
6    AAL 33N        LLLDDL  letters_digits_letters              2   
7    249 AAN        DDDLLL     digits_then_letters              2   
8   A114 ANA       LDDDLLL  letters_digits_letters              2   
9   A128 ART       LDDDLLL  letters_digits_letters              2   
10    188 AB         DDDLL     digits_then_letters              2   
11   1964 AB        DDDDLL     digits_then_letters              2   
12   ABA 21H        LLLDDL  letters_digits_letters              2   
13  AB06 BEY       LLDDLL

In [27]:
advanced_feature_summary = pd.Series(
    {
        "repeated_plate_rows": (
            len(repeated_plates)
        ),
        "unique_format_groups": (
            df_features[
                "plate_format_group"
            ].nunique()
        ),
        "all_digits_same": (
            df_features[
                "all_digits_same"
            ].sum()
        ),
        "all_letters_same": (
            df_features[
                "all_letters_same"
            ].sum()
        ),
        "double_character_runs": (
            df_features[
                "has_double_character"
            ].sum()
        ),
        "triple_character_runs": (
            df_features[
                "has_triple_character"
            ].sum()
        ),
        "four_character_runs": (
            df_features[
                "has_four_character_run"
            ].sum()
        ),
    },
    name="value"
)

advanced_feature_summary

Out[0]: 
repeated_plate_rows         6
unique_format_groups        3
all_digits_same          2406
all_letters_same          144
double_character_runs    6310
triple_character_runs    1032
four_character_runs        69
Name: value, dtype: int64


## 10. Target and Premium Labels

The modelling target is kept in both raw and log-transformed form. Premium labels are added using within-dataset price quantiles so later analysis can separate standard valuation from high-value plate identification.

In [28]:
df_features["log_hammer_price"] = np.log1p(
    df_features["hammer_price_gbp"]
)

premium_quantiles = {
    "top_10pct_threshold": df_features["hammer_price_gbp"].quantile(0.90),
    "top_5pct_threshold": df_features["hammer_price_gbp"].quantile(0.95),
    "top_1pct_threshold": df_features["hammer_price_gbp"].quantile(0.99),
}

for label, threshold in premium_quantiles.items():
    print(f"{label}: £{threshold:,.0f}")

df_features["is_premium_top_10pct"] = (
    df_features["hammer_price_gbp"]
    .ge(premium_quantiles["top_10pct_threshold"])
    .astype(int)
)

df_features["is_premium_top_5pct"] = (
    df_features["hammer_price_gbp"]
    .ge(premium_quantiles["top_5pct_threshold"])
    .astype(int)
)

df_features["is_premium_top_1pct"] = (
    df_features["hammer_price_gbp"]
    .ge(premium_quantiles["top_1pct_threshold"])
    .astype(int)
)


top_10pct_threshold: £5,690
top_5pct_threshold: £7,849
top_1pct_threshold: £14,020


## 11. Enhanced Interpretability Features

Additional features separate first and last numeric/letter blocks, capture short-plate scarcity, identify rounded numbers, and convert low-number desirability into a simple ordinal score.

In [29]:
def first_matching_block(blocks: list[str], pattern: str) -> str:
    for block in blocks:
        if re.fullmatch(pattern, block):
            return block
    return ""


def last_matching_block(blocks: list[str], pattern: str) -> str:
    for block in reversed(blocks):
        if re.fullmatch(pattern, block):
            return block
    return ""


df_features["first_numeric_block"] = (
    df_features["character_blocks"]
    .apply(lambda blocks: first_matching_block(blocks, r"\d+"))
)

df_features["last_numeric_block"] = (
    df_features["character_blocks"]
    .apply(lambda blocks: last_matching_block(blocks, r"\d+"))
)

df_features["first_letter_block"] = (
    df_features["character_blocks"]
    .apply(lambda blocks: first_matching_block(blocks, r"[A-Z]+"))
)

df_features["last_letter_block"] = (
    df_features["character_blocks"]
    .apply(lambda blocks: last_matching_block(blocks, r"[A-Z]+"))
)

df_features["first_numeric_value"] = pd.to_numeric(
    df_features["first_numeric_block"],
    errors="coerce"
)

df_features["last_numeric_value"] = pd.to_numeric(
    df_features["last_numeric_block"],
    errors="coerce"
)

df_features["first_numeric_length"] = (
    df_features["first_numeric_block"]
    .str.len()
)

df_features["last_numeric_length"] = (
    df_features["last_numeric_block"]
    .str.len()
)

df_features["first_letter_length"] = (
    df_features["first_letter_block"]
    .str.len()
)

df_features["last_letter_length"] = (
    df_features["last_letter_block"]
    .str.len()
)


In [30]:
df_features["is_short_plate_len_3_or_less"] = (
    df_features["plate_length"]
    .le(3)
    .astype(int)
)

df_features["is_short_plate_len_4_or_less"] = (
    df_features["plate_length"]
    .le(4)
    .astype(int)
)

df_features["is_round_number"] = (
    df_features["numeric_value"]
    .fillna(-1)
    .apply(
        lambda value: (
            value > 0
            and value % 10 == 0
        )
    )
    .astype(int)
)

df_features["is_milestone_number"] = (
    df_features["numeric_value"]
    .isin([1, 7, 8, 9, 10, 11, 22, 33, 44, 55, 66, 77, 88, 99, 100, 111, 222, 333, 444, 555, 666, 777, 888, 999, 1000, 1111, 7777, 8888])
    .astype(int)
)

def low_number_score(value: float) -> int:
    if pd.isna(value):
        return 0
    if value < 10:
        return 5
    if value < 100:
        return 4
    if value < 1_000:
        return 3
    if value < 10_000:
        return 2
    return 1


df_features["low_number_score"] = (
    df_features["numeric_value"]
    .apply(low_number_score)
)

# Keep validation strategy explicit for the next modelling notebook.
df_features["validation_group"] = df_features["event_code"]


In [31]:
enhanced_feature_summary = pd.Series(
    {
        "top_10pct_premium_rows": df_features["is_premium_top_10pct"].sum(),
        "top_5pct_premium_rows": df_features["is_premium_top_5pct"].sum(),
        "top_1pct_premium_rows": df_features["is_premium_top_1pct"].sum(),
        "short_len_3_or_less": df_features["is_short_plate_len_3_or_less"].sum(),
        "short_len_4_or_less": df_features["is_short_plate_len_4_or_less"].sum(),
        "round_numbers": df_features["is_round_number"].sum(),
        "milestone_numbers": df_features["is_milestone_number"].sum(),
        "validation_groups": df_features["validation_group"].nunique(),
    },
    name="value"
)

enhanced_feature_summary


Out[0]: 
top_10pct_premium_rows    1780
top_5pct_premium_rows      890
top_1pct_premium_rows      179
short_len_3_or_less          8
short_len_4_or_less       1266
round_numbers             2064
milestone_numbers         3963
validation_groups            9
Name: value, dtype: int64


## 12. Feature Table Export

The engineered dataset is written to a reusable processed CSV. This keeps the feature-engineering notebook as a real pipeline step rather than only an exploratory analysis.

In [32]:
FEATURE_OUTPUT = (
    PROCESSED_DATA_DIR
    / "plate_features_2025.csv"
)

required_feature_columns = [
    "plate_raw",
    "plate_clean",
    "plate_length",
    "letter_count",
    "digit_count",
    "numeric_value",
    "plate_pattern",
    "plate_format_group",
    "log_hammer_price",
    "is_premium_top_10pct",
    "is_premium_top_5pct",
    "is_premium_top_1pct",
    "validation_group",
]

missing_required_features = [
    column
    for column in required_feature_columns
    if column not in df_features.columns
]

assert not missing_required_features, (
    f"Missing required feature columns: {missing_required_features}"
)
assert len(df_features) == len(df), "Feature table row count changed unexpectedly."
assert df_features["hammer_price_gbp"].notna().all(), "Feature table contains missing targets."
assert df_features["plate_clean"].ne("").all(), "Feature table contains empty cleaned plates."
assert df_features["validation_group"].nunique() == df_features["event_code"].nunique(), (
    "Validation grouping no longer matches auction events."
)

df_features.to_csv(
    FEATURE_OUTPUT,
    index=False
)

print(f"Saved: {FEATURE_OUTPUT}")
print(f"Feature table rows: {len(df_features):,}")
print(f"Feature table columns: {df_features.shape[1]}")


Saved: /Users/osmanorka/Plate-Value-Intelligence/data/processed/plate_features_2025.csv
Feature table rows: 17,782
Feature table columns: 75


## 13. Feature Engineering Conclusion

This notebook converts the governed 2025 valuation dataset into a reusable feature table for modelling.

Key outcomes:

- Plate text is standardised while the original display text is preserved.
- Numeric, letter, repetition, sequence, segment, and format features are created.
- Premium labels are defined from 2025 price quantiles.
- `event_code` is retained as the validation group for later event-based model testing.
- The engineered feature table is exported to `data/processed/plate_features_2025.csv`.

The next project step can use this file directly for modelling without repeating feature engineering logic.